In [1]:
import pandas as pd

df = pd.DataFrame({
    "hours_studied": [20, 30, 40, 50, 60, 70, 80, 90, 100],
        "attendance": [20, 30, 40, 50, 60, 70, 80, 90, 100],
        "score": [25, 35, 45, 55, 65, 75, 85, 95, 100],
        "status": ["Fail","Fail","Fail","Fail","Pass","Pass","Pass","Pass","Pass"]
})
df

,hours_studied,attendance,score,status
0,20,20,25,Fail
1,30,30,35,Fail
2,40,40,45,Fail
3,50,50,55,Fail
4,60,60,65,Pass
5,70,70,75,Pass
6,80,80,85,Pass
7,90,90,95,Pass
8,100,100,100,Pass


### Binary Logistic Regression

In [2]:
import pandas as pd

df2 = pd.DataFrame({
    "hours_studied": [22, 34, 61, 77, 90],
        "attendance": [25, 35, 75, 85, 95],
        "score": [20, 30, 70, 80, 90],
        "status_probability": [0.2, 0.3, 0.6, 0.7, 0.89],
        "status": ["Fail","Fail","Pass","Pass","Pass"]
})

df2

,hours_studied,attendance,score,status_probability,status
0,22,25,20,0.20,Fail
1,34,35,30,0.30,Fail
2,61,75,70,0.60,Pass
3,77,85,80,0.70,Pass
4,90,95,90,0.89,Pass


In [3]:
df_titanic = pd.read_csv("Titanic-Dataset.csv")
print(df_titanic.head())
print(df_titanic.info())
print(df_titanic.describe())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

### DATA CLEANING

In [5]:
print(df_titanic.isna().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [6]:
print(df_titanic.duplicated().sum())

0


In [7]:
df_titanic = df_titanic.drop(["PassengerId","Cabin", "Name", "Ticket"], axis=1)
df_titanic["Age"] = df_titanic["Age"].fillna(df_titanic["Age"].median())
df_titanic["Embarked"] = df_titanic["Embarked"].fillna(df_titanic["Embarked"].mode()[0])
df_titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       891 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  891 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


### DATA PREPROCESSING

In [8]:
# Encoding categorical variables
df_titanic = pd.get_dummies(df_titanic, columns=["Sex", "Embarked"], drop_first=True)
df_titanic.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


In [10]:
X = df_titanic.drop("Survived", axis=1)
y = df_titanic["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
# Scaling features (optional but can improve performance)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  
X_test_scaled = scaler.transform(X_test)

In [17]:
# Training the logistic regression model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Predicting probabilities
y_prob = model.predict_proba(X_test_scaled)  # Probability of survival
print("Predicted probabilities:\n", y_prob[:5])
# Making predictions
y_pred = model.predict(X_test_scaled)
print("Predicted classes:\n", y_pred[:5])

predicted_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)
predicted_df['Actual_Survived'] = y_test.values
predicted_df['Predicted_Survived'] = y_pred
predicted_df['Survival_Probability'] = y_prob[:, 1]  # Probability of survival
print(predicted_df.head())

Predicted probabilities:
 [[0.93374384 0.06625616]
 [0.95342876 0.04657124]
 [0.84840028 0.15159972]
 [0.96520361 0.03479639]
 [0.31638207 0.68361793]]
Predicted classes:
 [0 0 0 0 1]
     Pclass       Age     SibSp     Parch      Fare  Sex_male  Embarked_Q  \
0  0.829568 -0.419170  1.421753 -0.466183 -0.159704  0.742427   -0.289333   
1  0.829568  1.116290 -0.465084  0.727782 -0.327324  0.742427   -0.289333   
2  0.829568 -0.572716 -0.465084 -0.466183 -0.512122  0.742427   -0.289333   
3  0.829568  0.885971  1.421753 -0.466183 -0.368795  0.742427   -0.289333   
4  0.829568 -0.112078  0.478335 -0.466183 -0.339817 -1.346933    3.456220   

   Embarked_S  Actual_Survived  Predicted_Survived  Survival_Probability  
0    0.611978                0                   0              0.066256  
1    0.611978                0                   0              0.046571  
2   -1.634045                1                   0              0.151600  
3    0.611978                0                   0   

In [19]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)

class_report = classification_report(y_test, y_pred)
print("Classification Report:\n", class_report)

Accuracy: 0.8044692737430168
Confusion Matrix:
 [[98 12]
 [23 46]]
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

